In [1]:
import pandas as pd
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus.common import thai_stopwords

stopwords = set(thai_stopwords())  # ← CALL the function to get the set

def preprocess_text(text):
    tokens = word_tokenize(str(text))
    return [word for word in tokens if word not in stopwords]

# Load Excel data
file_path = "data_course_for_RSv2.xlsx"
scraped_jobs_path = "combined_all_jobs_requirements.xlsx"
courses = pd.read_excel(file_path, sheet_name="Sheet1")
jobs = pd.read_excel(scraped_jobs_path)

# Rename columns for jobs
jobs = jobs.rename(columns={
    'Job Title': 'JOB_NAME',
    'Job Description': 'REQUIREMENTS'
})

# Apply preprocessing
courses['PROCESSED_DESC'] = courses['C_Description'].apply(preprocess_text)
jobs['PROCESSED_REQ'] = jobs['REQUIREMENTS'].apply(preprocess_text)

# Tagging documents for Doc2Vec
tagged_docs = []

# Tag courses as C_0, C_1, ...
for i, tokens in enumerate(courses['PROCESSED_DESC']):
    tagged_docs.append(TaggedDocument(tokens, [f"C_{i}"]))

# Tag jobs as J_0, J_1, ...
for i, tokens in enumerate(jobs['PROCESSED_REQ']):
    tagged_docs.append(TaggedDocument(tokens, [f"J_{i}"]))

# Train Doc2Vec model
doc2vec_model = Doc2Vec(vector_size=300, window=5, min_count=1, workers=4, epochs=40)
doc2vec_model.build_vocab(tagged_docs)
doc2vec_model.train(tagged_docs, total_examples=doc2vec_model.corpus_count, epochs=doc2vec_model.epochs)

# Infer vectors for course descriptions
courses['EMBEDDING'] = courses['PROCESSED_DESC'].apply(lambda x: doc2vec_model.infer_vector(x))

# Infer vectors for job descriptions
jobs['EMBEDDING'] = jobs['PROCESSED_REQ'].apply(lambda x: doc2vec_model.infer_vector(x))

In [2]:
learner_profile = {
    "desired_job": "นักพัฒนาซอฟต์แวร์",
    "courses_completed": [
        "คอมพิวเตอร์สารสนเทศขั้นพื้นฐาน",
        "ภาษาอังกฤษเพื่อการสื่อสาร",
        "วิทยาการคอมพิวเตอร์พื้นฐาน",
        "โครงงานวิทยาการคอมพิวเตอร์",
        "คอมพิวเตอร์กราฟิก",
        "ภาษาอังกฤษสำหรับวิชาชีพคอมพิวเตอร์",
        "การเขียนโปรแกรมขั้นพื้นฐาน",
        "การเขียนโปรแกรมเชิงวัตถุ",
        "การพัฒนาซอฟต์แวร์เชิงวัตถุ",
        "แคลคูลัสและเรขาคณิตวิเคราะห์ 1",
    ]
}

In [8]:
# Get the vector for the desired job
desired_job_vector = jobs[jobs['JOB_NAME'] == (learner_profile['desired_job'])]['EMBEDDING'].iloc[0]

# Compute similarity
candidates = courses[~courses["C_Name"].isin(learner_profile['courses_completed'])].copy()

from sklearn.metrics.pairwise import cosine_similarity
courses['SIMILARITY'] = courses['EMBEDDING'].apply(
    lambda x: cosine_similarity([desired_job_vector], [x])[0][0])

# Top course recommendations
candidates_unique = candidates.drop_duplicates(subset="C_Name", keep="first")
candidates_sorted = candidates_unique.sort_values(['SIMILARITY'], ascending=False).head(10)


# Output recommendations
print("Your desired job is:")
print (learner_profile['desired_job'])
print("\nRecommended Courses:")
print(candidates_sorted[['C_Name', 'C_Description', 'SIMILARITY']])

Your desired job is:
นักพัฒนาซอฟต์แวร์

Recommended Courses:
                                        C_Name  \
44  การฝึกประสบการณ์วิชาชีพวิทยาการคอมพิวเตอร์   
10                      การเขียนโปรแกรมขั้นสูง   
22   การปฏิสัมพันธ์ระหว่างมนุษย์กับคอมพิวเตอร์   
42               การเขียนโปรแกรมประยุกต์บนเว็บ   
9             หลักการและระบบการจัดการฐานข้อมูล   
38                           วิศวกรรมซอฟต์แวร์   
33                   การวิเคราะห์และออกแบบระบบ   
26                      ภาษาไทยเพื่อการสื่อสาร   
36        ประเด็นคัดสรรด้านวิทยาการคอมพิวเตอร์   
83  การวางแผนและการประเมินโครงการด้านสาธารณสุข   

                                        C_Description  SIMILARITY  
44  นักศึกษาฝึกประสบการณ์วิชาชีพวิทยาการคอมพิวเตอร...    0.967029  
10  แนวคิดในการเขียนโปรแกรมด้วยภาษาคอมพิวเตอร์ระดั...    0.894046  
22   หลักการการปฏิสัมพันธ์ระหว่างมนุษย์กับคอมพิวเต...    0.886120  
42  เทคโนโลยีอินเทอร์เน็ต เทคโนโลยีเว็บ สถาปัตยกรร...    0.865595  
9    ระบบแฟ้มข้อมูล ระบบฐานข้อมูล องค์ประกอบและส